# RoadSafe Analytics
## 02 — Data Cleaning & Preprocessing

### Objective
Clean and preprocess the collision, vehicle, and casualty datasets while preserving important information for EDA, regression, classification, and clustering.

### Cleaning Tasks
1. Identify and handle coded missing values. (using official STATS19 code definitions)
2. Review deprecated and historical variables.
3. Validate numerical and categorical values.
4. Convert dates, times, and data types.
5. Preserve collision and vehicle/casualty identifiers.
6. Preserve classification and regression targets.
7. Compare data quality before and after cleaning.
8. Export cleaned datasets for Stage 5.

In [1]:
import pandas as pd 
import numpy as np
from pathlib import Path

In [2]:
project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

raw_dir = project_root / "data" / "raw"
processed_dir = project_root / "data" / "processed"

print("Project root:", project_root)
print("Raw data exists:", raw_dir.exists())
print("Processed directory exists:", processed_dir.exists())

Project root: C:\Users\Tanishka Singh\OneDrive\Desktop\UPES\Semester-3\FDS\Project\RoadSafe-Analytics
Raw data exists: True
Processed directory exists: True


In [3]:
collisions_raw = pd.read_csv(raw_dir / "dft-road-casualty-statistics-collision-2025.csv",
    dtype={"collision_index": "string", "collision_ref_no": "string"}, low_memory=False)

vehicles_raw = pd.read_csv(raw_dir / "dft-road-casualty-statistics-vehicle-2025.csv",
    dtype={"collision_index": "string", "collision_ref_no": "string"}, low_memory=False)

casualties_raw = pd.read_csv(raw_dir / "dft-road-casualty-statistics-casualty-2025.csv",
    dtype={"collision_index": "string", "collision_ref_no": "string"}, low_memory=False)

print("Collisions:", collisions_raw.shape)
print("Vehicles:", vehicles_raw.shape)
print("Casualties:", casualties_raw.shape)

Collisions: (101525, 44)
Vehicles: (183948, 32)
Casualties: (127883, 24)


In [4]:
collisions = collisions_raw.copy()
vehicles = vehicles_raw.copy()
casualties = casualties_raw.copy()

print("Working copies created successfully!")

Working copies created successfully!


## Initial Data Quality Assessment

Before cleaning, we examine the original datasets to identify missing values, duplicate records, and structural issues.

These results provide a baseline for evaluating the effectiveness of our cleaning process.

In [5]:
datasets_raw = {"Collisions": collisions_raw, "Vehicles": vehicles_raw, "Casualties": casualties_raw}

baseline_summary = []

for name, df in datasets_raw.items():
    baseline_summary.append({
        "Dataset": name,
        "Rows": len(df),
        "Columns": df.shape[1],
        "Missing Cells": df.isna().sum().sum(),
        "Duplicate Rows": df.duplicated().sum(),
        "Duplicate Collision IDs": df["collision_index"].duplicated().sum()
    })

baseline_df = pd.DataFrame(baseline_summary)

display(baseline_df)

,Dataset,Rows,Columns,Missing Cells,Duplicate Rows,Duplicate Collision IDs
0,Collisions,101525,44,8,0,0
1,Vehicles,183948,32,0,0,82423
2,Casualties,127883,24,0,0,26358


## Investigation of Coded Missing Values

STATS19 uses special numerical codes to represent unavailable, unknown, deprecated, or inapplicable information.

We first examine the frequency of `-1` across the three datasets. These values will be interpreted using the official STATS19 data dictionary before any cleaning decisions are made.

In [6]:
for name, df in datasets_raw.items():
    minus_one_counts = (df == -1).sum()
    minus_one_counts = minus_one_counts[minus_one_counts > 0]

    result = pd.DataFrame({
        "Count": minus_one_counts,
        "Percentage (%)": (minus_one_counts / len(df) * 100).round(2)
    })

    print(f"\n{name} Dataset")
    print("-" * 45)

    display(result.sort_values("Count", ascending=False))


Collisions Dataset
---------------------------------------------


,Count,Percentage (%)
local_authority_district,101525.0,100.0
junction_detail_historic,87871.0,86.55
special_conditions_at_site,87858.0,86.54
pedestrian_crossing_human_control_historic,87852.0,86.53
carriageway_hazards_historic,87852.0,86.53
pedestrian_crossing_physical_facilities_historic,87851.0,86.53
junction_control,44487.0,43.82
second_road_number,25105.0,24.73
enhanced_severity_collision,13677.0,13.47
trunk_road_flag,7108.0,7.0



Vehicles Dataset
---------------------------------------------


,Count,Percentage (%)
age_of_vehicle,183948.0,100.0
vehicle_manoeuvre_historic,161547.0,87.82
journey_purpose_of_driver_historic,161056.0,87.56
vehicle_location_restricted_lane_historic,159544.0,86.73
engine_capacity_cc,44869.0,24.39
driver_imd_decile,41539.0,22.58
driver_distance_banding,41539.0,22.58
propulsion_code,38869.0,21.13
age_of_driver,28262.0,15.36
age_band_of_driver,28262.0,15.36



Casualties Dataset
---------------------------------------------


,Count,Percentage (%)
pedestrian_road_maintenance_worker,17457.0,13.65
enhanced_casualty_severity,17394.0,13.6
casualty_imd_decile,14663.0,11.47
casualty_distance_banding,14663.0,11.47
age_of_casualty,2741.0,2.14
age_band_of_casualty,2741.0,2.14
sex_of_casualty,1320.0,1.03
car_passenger,500.0,0.39
bus_or_coach_passenger,282.0,0.22
pedestrian_location,43.0,0.03


### Official STATS19 Data Dictionary Verification

The official STATS19 data dictionary is used to interpret special categorical codes before applying cleaning rules.

We will distinguish unavailable, unknown, not applicable, and deprecated values to avoid removing meaningful information.

In [7]:
dictionary_files = list(raw_dir.glob("*.xlsx"))

print("Available Excel files:")
for file in dictionary_files:
    print(file.name)

Available Excel files:
dft-road-casualty-statistics-road-safety-open-dataset-data-guide-2025.xlsx


In [8]:
data_guide_path = dictionary_files[0]

excel_file = pd.ExcelFile(data_guide_path)

print("Available sheets:")
print(excel_file.sheet_names)

code_list = pd.read_excel(data_guide_path, sheet_name="2024_code_list")

display(code_list.head(10))

Available sheets:
['2024_code_list', '2011_to_2024_conversion']


,table,field name,code/format,label,note
0,collision,collision_index,NaN,NaN,unique value for each collision. The collision...
1,collision,collision_year,NaN,NaN,NaN
2,collision,collision_ref_no,NaN,NaN,In year id used by the police to reference a c...
3,collision,location_easting_osgr,NaN,NaN,Null if not known
4,collision,location_northing_osgr,NaN,NaN,Null if not known
5,collision,longitude,NaN,NaN,Null if not known
6,collision,latitude,NaN,NaN,Null if not known
7,collision,police_force,1,Metropolitan Police,NaN
8,collision,police_force,3,Cumbria,NaN
9,collision,police_force,4,Lancashire,NaN


### Interpretation of -1 Codes

The meaning of -1 varies across STATS19 variables.

We examine the official data dictionary to distinguish missing, deprecated, unknown, and inapplicable values before applying cleaning rules.

In [9]:
minus_one_definitions = code_list[code_list["code/format"].astype(str).str.strip() == "-1"]

print("Total -1 definitions:", len(minus_one_definitions))

display(minus_one_definitions[["table", "field name", "code/format", "label", "note"]])

Total -1 definitions: 63


,table,field name,code/format,label,note
67,collision,enhanced_collision_severity,-1,Data missing or out of range,category introduced where collected in 2023
611,collision,local_authority_district,-1,Code deprecated,NaN
1034,collision,local_authority_ons_district,-1,Record predates use of local_authority_ons_dis...,NaN
1254,collision,local_authority_highway,-1,Record predates use of local_authority_highway...,NaN
1262,collision,first_road_class,-1,Data missing or out of range,NaN
...,...,...,...,...,...
1753,casualty,bus_or_coach_passenger,-1,Data missing or out of range,NaN
1758,casualty,pedestrian_road_maintenance_worker,-1,Data missing or out of range,NaN
1790,casualty,casualty_type,-1,Data missing or out of range,NaN
1801,casualty,casualty_imd_decile,-1,Data missing or out of range,field introduced in 2016


### Matching Coded Values with Official Definitions

We compare the observed -1 values in the three STATS19 datasets with their definitions in the official data guide.

This helps identify deprecated fields, unavailable information, and values requiring special treatment during preprocessing.

In [10]:
table_names = {"Collisions": "collision", "Vehicles": "vehicle", "Casualties": "casualty"}

observed_codes = []

for name, df in datasets_raw.items():
    counts = (df == -1).sum()

    for column, count in counts[counts > 0].items():
        observed_codes.append({
            "table": table_names[name],
            "field name": column,
            "count": int(count),
            "percentage": round(count / len(df) * 100, 2)
        })

observed_df = pd.DataFrame(observed_codes)

definitions = minus_one_definitions[["table", "field name", "label", "note"]].copy()

cleaning_review = observed_df.merge(definitions, on=["table", "field name"], how="left", validate="one_to_one")

display(cleaning_review)

,table,field name,count,percentage,label,note
0,collision,local_authority_district,101525,100.00,Code deprecated,NaN
1,collision,first_road_class,6,0.01,Data missing or out of range,NaN
2,collision,first_road_number,6,0.01,Unknown,NaN
3,collision,junction_detail_historic,87871,86.55,Data missing or out of range,data collected in the 2011 STATS19 specificati...
4,collision,junction_detail,6701,6.60,Data missing or out of range,New field added with the 2024 data - data coll...
5,collision,junction_control,44487,43.82,Data missing or out of range,NaN
6,collision,second_road_class,171,0.17,Data missing or out of range,NaN
7,collision,second_road_number,25105,24.73,Unknown,NaN
8,collision,pedestrian_crossing_human_control_historic,87852,86.53,Data missing or out of range,data collected in the 2011 STATS19 specificati...
9,collision,pedestrian_crossing_physical_facilities_historic,87851,86.53,Data missing or out of range,data collected in the 2011 STATS19 specificati...


### Review of Unmatched Dictionary Definitions

Some fields containing coded missing values may not have matching definitions in the official data dictionary.

These fields are examined separately before cleaning decisions are made.

In [11]:
unmatched_codes = cleaning_review[cleaning_review["label"].isna()]
display(unmatched_codes)

,table,field name,count,percentage,label,note
17,collision,enhanced_severity_collision,13677,13.47,NaN,NaN
40,vehicle,age_of_vehicle,183948,100.00,NaN,NaN
42,vehicle,driver_distance_banding,41539,22.58,NaN,field introduced in 2023


### Investigation of Fields with Incomplete Code Definitions

Three variables require additional examination because their -1 codes have missing labels or possible naming inconsistencies.

We inspect their available dictionary entries before making cleaning decisions.

In [12]:
fields_to_check = ["enhanced_severity_collision","enhanced_collision_severity","age_of_vehicle","driver_distance_banding"]
field_definitions = code_list[code_list["field name"].isin(fields_to_check)]

display(field_definitions[["table", "field name", "code/format", "label", "note"]])

,table,field name,code/format,label,note
62,collision,enhanced_collision_severity,1,Fatal,category introduced where collected in 2023
63,collision,enhanced_collision_severity,5,Very Serious,category introduced where collected in 2023
64,collision,enhanced_collision_severity,6,Moderately Serious,category introduced where collected in 2023
65,collision,enhanced_collision_severity,7,Less Serious,category introduced where collected in 2023
66,collision,enhanced_collision_severity,3,Slight,category introduced where collected in 2023
67,collision,enhanced_collision_severity,-1,Data missing or out of range,category introduced where collected in 2023
1661,vehicle,age_of_vehicle,NaN,NaN,NaN
1678,vehicle,driver_distance_banding,1,Collision occurred within 5km of drivers home ...,field introduced in 2023
1679,vehicle,driver_distance_banding,2,Collision occurred between 5.001 and 10km of d...,field introduced in 2023
1680,vehicle,driver_distance_banding,3,Collision occurred between 10.001 and 20km of ...,field introduced in 2023


## Data Cleaning and Preprocessing

### Removal of Unusable Columns

Two variables were identified as unsuitable for analysis:

- `local_authority_district`: Deprecated and contains -1 in every collision record.
- `age_of_vehicle`: Contains -1 in every vehicle record and provides no useful variation.

These columns are removed from the working datasets.
The original raw datasets remain unchanged.

In [13]:
collisions = collisions.drop(columns=["local_authority_district"])
vehicles = vehicles.drop(columns=["age_of_vehicle"])

print("Collisions:", collisions.shape)
print("Vehicles:", vehicles.shape)
print("Casualties:", casualties.shape)

Collisions: (101525, 43)
Vehicles: (183948, 31)
Casualties: (127883, 24)


### Handling Coded Missing Values

The STATS19 datasets contain special numerical codes such as -1 that represent missing, unknown, undefined, or out-of-range information.

Using the official data dictionary, these codes are converted to Pandas missing values where appropriate.

No records are deleted, and the original raw datasets remain unchanged.

In [14]:
missing_labels = ["data missing","unknown","undefined"]

missing_definitions = cleaning_review[cleaning_review["label"].fillna("").str.lower().str.contains("|".join(missing_labels), regex=True)]

missing_fields = {table: group["field name"].tolist()for table, group in missing_definitions.groupby("table")}

for table, df in [("collision", collisions),("vehicle", vehicles),("casualty", casualties)]:
    columns = missing_fields.get(table, [])

    for column in columns:
        df[column] = df[column].replace(-1, np.nan)

    print(f"{table}: converted -1 codes in {len(columns)} columns")

collision: converted -1 codes in 16 columns
vehicle: converted -1 codes in 23 columns
casualty: converted -1 codes in 12 columns


### Validation of Coded Missing-Value Conversion

After converting documented unavailable codes to missing values, we examine the remaining -1 values and compare missing-value counts with the original datasets.

This ensures that the cleaning operation is transparent and that unresolved special codes are not accidentally treated as valid numerical measurements.

In [15]:
datasets_clean = {"Collisions": collisions,"Vehicles": vehicles,"Casualties": casualties}

for name, df in datasets_clean.items():
    print(f"\n{name} Dataset")
    print("-" * 45)

    print("Shape:", df.shape)
    print("Missing cells:", df.isna().sum().sum())

    remaining = (df == -1).sum()
    remaining = remaining[remaining > 0]

    print("\nColumns still containing -1:")
    if remaining.empty:
        print("None")
    else:
        display(remaining.sort_values(ascending=False))


Collisions Dataset
---------------------------------------------
Shape: (101525, 43)
Missing cells: 535399

Columns still containing -1:


enhanced_severity_collision    13677.0
dtype: double[pyarrow]


Vehicles Dataset
---------------------------------------------
Shape: (183948, 31)
Missing cells: 722196

Columns still containing -1:


driver_distance_banding    41539.0
dtype: double[pyarrow]


Casualties Dataset
---------------------------------------------
Shape: (127883, 24)
Missing cells: 71847

Columns still containing -1:
None


### Handling Remaining Unavailable Codes

Two fields retain -1 values after the initial conversion:

- `enhanced_severity_collision`: Uses a different field name from
  `enhanced_collision_severity` in the official code list.
- `driver_distance_banding`: Valid distance categories are coded
  1–5, while -1 has no defined distance category.

For this analysis, -1 is treated as unavailable in these two fields. No records are removed.

In [16]:
collisions["enhanced_severity_collision"] = (collisions["enhanced_severity_collision"].replace(-1, np.nan))

vehicles["driver_distance_banding"] = (vehicles["driver_distance_banding"].replace(-1, np.nan))

print("Remaining -1 in enhanced severity:", (collisions["enhanced_severity_collision"] == -1).sum())

print("Remaining -1 in driver distance:", (vehicles["driver_distance_banding"] == -1).sum())

Remaining -1 in enhanced severity: 0
Remaining -1 in driver distance: 0


## Review of Historical Variables

Some STATS19 variables have historical versions from older reporting specifications.

We examine these fields and their newer equivalents to determine whether the historical columns provide additional information or can be excluded from the analysis-ready datasets.

In [17]:
for name, df in datasets_clean.items():
    historic_columns = [col for col in df.columns if col.endswith("_historic")]

    print(f"\n{name} Dataset")
    print("Historical columns:", historic_columns)

    if historic_columns:
        display(pd.DataFrame({
                "Missing (%)": (df[historic_columns].isna().mean() * 100).round(2),
                "Unique non-missing values": (df[historic_columns].nunique(dropna=True))
            })
        )


Collisions Dataset
Historical columns: ['junction_detail_historic', 'pedestrian_crossing_human_control_historic', 'pedestrian_crossing_physical_facilities_historic', 'carriageway_hazards_historic']


,Missing (%),Unique non-missing values
junction_detail_historic,86.55,10
pedestrian_crossing_human_control_historic,86.53,4
pedestrian_crossing_physical_facilities_historic,86.53,7
carriageway_hazards_historic,86.53,7



Vehicles Dataset
Historical columns: ['vehicle_manoeuvre_historic', 'vehicle_location_restricted_lane_historic', 'journey_purpose_of_driver_historic']


,Missing (%),Unique non-missing values
vehicle_manoeuvre_historic,87.82,19
vehicle_location_restricted_lane_historic,86.73,11
journey_purpose_of_driver_historic,87.56,6



Casualties Dataset
Historical columns: []


### Comparison of Historical and Current Variables

Historical STATS19 variables may contain information recorded under older reporting specifications.

We compare the availability of historical and current fields before deciding whether to exclude the historical variables.

In [18]:
historical_pairs = {
    "Collisions": [("junction_detail_historic", "junction_detail"), 
                   ("carriageway_hazards_historic", "carriageway_hazards")],
    "Vehicles": [("vehicle_manoeuvre_historic", "vehicle_manoeuvre"),
                 ("vehicle_location_restricted_lane_historic","vehicle_location_restricted_lane"),
                 ("journey_purpose_of_driver_historic","journey_purpose_of_driver")]
}

comparison_results = []

for name, pairs in historical_pairs.items():
    df = datasets_clean[name]

    for historic, current in pairs:
        comparison_results.append({
            "Dataset": name,
            "Historical Field": historic,
            "Current Field": current,
            "Historical Only": int((df[historic].notna() & df[current].isna()).sum()),
            "Current Only": int((df[historic].isna() & df[current].notna()).sum()),
            "Both Available": int((df[historic].notna() & df[current].notna()).sum()),
            "Both Missing": int((df[historic].isna() & df[current].isna()).sum())
        })

comparison_df = pd.DataFrame(comparison_results)

display(comparison_df)

,Dataset,Historical Field,Current Field,Historical Only,Current Only,Both Available,Both Missing
0,Collisions,junction_detail_historic,junction_detail,275,81445,13379,6426
1,Collisions,carriageway_hazards_historic,carriageway_hazards,0,82506,13673,5346
2,Vehicles,vehicle_manoeuvre_historic,vehicle_manoeuvre,0,156349,22401,5198
3,Vehicles,vehicle_location_restricted_lane_historic,vehicle_location_restricted_lane,0,154176,24404,5368
4,Vehicles,journey_purpose_of_driver_historic,journey_purpose_of_driver,0,156661,22892,4395


## Investigation of Other Special Codes

Apart from -1, STATS19 uses other numerical codes to represent unknown, unavailable, or undefined information.

We examine the official data dictionary to identify these codes before deciding how to handle them.

Valid categorical values will be preserved.

In [19]:
special_keywords = r"unknown|missing|undefined|not known|not recorded"

special_codes = code_list[code_list["label"].fillna("").astype(str).str.contains(special_keywords, case=False, regex=True)].copy()

special_codes = special_codes[special_codes["code/format"].notna()]

display(special_codes[["table", "field name", "code/format", "label"]])

,table,field name,code/format,label
67,collision,enhanced_collision_severity,-1,Data missing or out of range
1262,collision,first_road_class,-1,Data missing or out of range
1264,collision,first_road_number,-1,Unknown
1271,collision,road_type,9,Unknown
1273,collision,road_type,-1,Data missing or out of range
...,...,...,...,...
1780,casualty,casualty_type,98,Goods vehicle (unknown weight) occupant
1781,casualty,casualty_type,99,Unknown vehicle type (self rep only)
1790,casualty,casualty_type,-1,Data missing or out of range
1801,casualty,casualty_imd_decile,-1,Data missing or out of range


In [20]:
dataset_lookup = {"collision": collisions,"vehicle": vehicles,"casualty": casualties}

special_code_counts = []

for _, row in special_codes.iterrows():
    table = row["table"]
    column = row["field name"]
    code = row["code/format"]

    if table not in dataset_lookup:
        continue

    df = dataset_lookup[table]

    if column not in df.columns:
        continue

    count = (df[column].astype("string").str.strip()== str(code).strip()).sum()

    if count > 0:
        special_code_counts.append({"Dataset": table,"Column": column,"Code": code,"Meaning": row["label"],"Count": int(count)})

special_code_summary = pd.DataFrame(special_code_counts)

display(special_code_summary)

,Dataset,Column,Code,Meaning,Count
0,collision,road_type,9,Unknown,1953
1,collision,weather_conditions,9,Unknown,2385
2,vehicle,generic_make_model,-1,Data missing or out of range,37531


### Handling Additional Special Codes

The official STATS19 dictionary identifies additional unknown
and unavailable values in road type, weather conditions, and vehicle make/model.

These values are converted to missing values while preserving all collision and vehicle records.

In [21]:
collisions["road_type"] = collisions["road_type"].replace(9, np.nan)
collisions["weather_conditions"] = (collisions["weather_conditions"].replace(9, np.nan))
vehicles["generic_make_model"] = (vehicles["generic_make_model"].replace([-1, "-1"], pd.NA))

print("Unknown road types:",(collisions["road_type"] == 9).sum())
print("Unknown weather conditions:",(collisions["weather_conditions"] == 9).sum())

print("Unavailable vehicle models:",vehicles["generic_make_model"].astype("string").str.strip().eq("-1").fillna(False).sum())

Unknown road types: 0
Unknown weather conditions: 0
Unavailable vehicle models: 0


## Date and Time Cleaning

Collision dates and times are validated to ensure they follow consistent formats.

Dates are converted to datetime objects, while times are checked for valid 24-hour formatting.

Invalid values are identified without deleting records.

In [22]:
# Convert collision dates
collisions["date"] = pd.to_datetime(collisions["date"],format="%d/%m/%Y",errors="coerce")

# Validate time values
parsed_times = pd.to_datetime(collisions["time"],format="%H:%M",errors="coerce")

# Count invalid or missing values
print("Invalid or missing dates:", collisions["date"].isna().sum())
print("Invalid or missing times:", parsed_times.isna().sum())

# Check date range
print("Earliest collision:", collisions["date"].min())
print("Latest collision:", collisions["date"].max())

# Display sample values
display(collisions[["date", "time"]].head())

Invalid or missing dates: 0
Invalid or missing times: 0
Earliest collision: 2025-01-01 00:00:00
Latest collision: 2025-12-31 00:00:00


,date,time
0,2025-02-04,15:00
1,2025-05-07,09:35
2,2025-11-27,18:18
3,2025-05-15,16:30
4,2025-12-08,07:45


## Numerical and Geographical Validation

Important numerical attributes are checked for invalid or out-of-range values.

These checks help identify inconsistent records without automatically removing observations.

In [23]:
validation_results = {"Invalid collision severity": (~collisions["collision_severity"].isin([1, 2, 3])).sum(),
                      "Invalid vehicle counts": (collisions["number_of_vehicles"] < 1).sum(),
                      "Invalid casualty counts": (collisions["number_of_casualties"] < 1).sum(),
                      "Invalid speed limits": ((collisions["speed_limit"] <= 0) |(collisions["speed_limit"] > 70)).sum(),
                      "Invalid latitude": (collisions["latitude"].notna() & ~collisions["latitude"].between(-90, 90)).sum(),
                      "Invalid longitude": (collisions["longitude"].notna() & ~collisions["longitude"].between(-180, 180)).sum(),
                      "Invalid driver ages": (vehicles["age_of_driver"].notna() & ~vehicles["age_of_driver"].between(0, 120)).sum(),
                      "Invalid casualty ages": (casualties["age_of_casualty"].notna() & ~casualties["age_of_casualty"].between(0, 120)).sum()
                      }

validation_df = pd.DataFrame(validation_results.items(),columns=["Validation Check", "Invalid Records"])

display(validation_df)

,Validation Check,Invalid Records
0,Invalid collision severity,0
1,Invalid vehicle counts,0
2,Invalid casualty counts,0
3,Invalid speed limits,0
4,Invalid latitude,0
5,Invalid longitude,0
6,Invalid driver ages,0
7,Invalid casualty ages,0


## Final Dataset Integrity Validation

Before exporting the cleaned datasets, we verify that:

- No records were unintentionally deleted.
- Collision identifiers remain unique in the collision dataset.
- Vehicle and casualty records retain valid references.
- Composite identifiers remain unique.
- Important prediction targets are preserved.

In [24]:
integrity_checks = {"Collision row count preserved":len(collisions) == len(collisions_raw),
                    "Vehicle row count preserved":len(vehicles) == len(vehicles_raw),
                    "Casualty row count preserved":len(casualties) == len(casualties_raw),
                    "Collision IDs unique": collisions["collision_index"].is_unique,
                    "Vehicle composite keys unique": not vehicles.duplicated(subset=["collision_index", "vehicle_reference"]).any(),
                    "Casualty composite keys unique":not casualties.duplicated(subset=["collision_index", "casualty_reference"]).any(),
                    "Vehicle collision references valid":vehicles["collision_index"].isin(collisions["collision_index"]).all(),
                    "Casualty collision references valid":casualties["collision_index"].isin(collisions["collision_index"]).all(),
                    "Casualty vehicle references valid":pd.MultiIndex.from_frame(
                        casualties[["collision_index", "vehicle_reference"]])
                        .isin(pd.MultiIndex.from_frame(vehicles[["collision_index", "vehicle_reference"]])).all(),
                    "Collision severity target preserved":collisions["collision_severity"].equals(collisions_raw["collision_severity"]),
                    "Casualty count target preserved":collisions["number_of_casualties"].equals(collisions_raw["number_of_casualties"])
                    }

integrity_df = pd.DataFrame({"Check": integrity_checks.keys(),"Passed": integrity_checks.values()})

display(integrity_df)

print("All integrity checks passed:", all(integrity_checks.values()))

,Check,Passed
0,Collision row count preserved,True
1,Vehicle row count preserved,True
2,Casualty row count preserved,True
3,Collision IDs unique,True
4,Vehicle composite keys unique,True
5,Casualty composite keys unique,True
6,Vehicle collision references valid,True
7,Casualty collision references valid,True
8,Casualty vehicle references valid,True
9,Collision severity target preserved,True


All integrity checks passed: True


## Final Data Quality Summary

The cleaned datasets are compared with their original versions to summarize changes in missing values, dimensions, and duplicate
records.

Missing values are retained where information is genuinely unavailable rather than removing entire records.

In [25]:
quality_summary = []

for name in datasets_raw:
    original = datasets_raw[name]
    cleaned = datasets_clean[name]

    quality_summary.append({
        "Dataset": name,
        "Original Rows": len(original),
        "Cleaned Rows": len(cleaned),
        "Original Columns": original.shape[1],
        "Cleaned Columns": cleaned.shape[1],
        "Original Missing Cells": int(original.isna().sum().sum()),
        "Cleaned Missing Cells": int(cleaned.isna().sum().sum()),
        "Cleaned Duplicate Rows": int(cleaned.duplicated().sum())
    })

quality_summary_df = pd.DataFrame(quality_summary)

display(quality_summary_df)

,Dataset,Original Rows,Cleaned Rows,Original Columns,Cleaned Columns,Original Missing Cells,Cleaned Missing Cells,Cleaned Duplicate Rows
0,Collisions,101525,101525,44,43,8,553414,0
1,Vehicles,183948,183948,32,31,0,801266,0
2,Casualties,127883,127883,24,24,0,71847,0


## Export Cleaned Datasets

The cleaned collision, vehicle, and casualty datasets are exported in Parquet format for subsequent data integration, exploratory analysis, and machine learning.

The original raw CSV files remain unchanged.

In [26]:
processed_dir.mkdir(parents=True, exist_ok=True)

collisions.to_parquet(processed_dir / "collisions_clean.parquet",index=False)
vehicles.to_parquet(processed_dir / "vehicles_clean.parquet",index=False)
casualties.to_parquet(processed_dir / "casualties_clean.parquet",index=False)

print("All three cleaned datasets exported successfully!")

All three cleaned datasets exported successfully!


In [27]:
for name in [
    "collisions_clean.parquet",
    "vehicles_clean.parquet",
    "casualties_clean.parquet"
]:
    path = processed_dir / name
    print(name, "| Exists:", path.exists())

collisions_clean.parquet | Exists: True
vehicles_clean.parquet | Exists: True
casualties_clean.parquet | Exists: True


In [28]:
exported_datasets = {
    "Collisions": ("collisions_clean.parquet", collisions),
    "Vehicles": ("vehicles_clean.parquet", vehicles),
    "Casualties": ("casualties_clean.parquet", casualties)
}

for name, (filename, original_df) in exported_datasets.items():
    saved_df = pd.read_parquet(processed_dir / filename)

    print(
        f"{name}:",
        f"Rows = {len(saved_df)}",
        f"Columns = {saved_df.shape[1]}",
        f"Matches working data = {saved_df.equals(original_df)}"
    )

Collisions: Rows = 101525 Columns = 43 Matches working data = True
Vehicles: Rows = 183948 Columns = 31 Matches working data = True
Casualties: Rows = 127883 Columns = 24 Matches working data = True
